# Customer Support Ticket Routing

Route short support messages to **billing**, **technical**, or **shipping** teams.

**Use case:** Reduce triage time in help desks and chatbots.

**Prerequisites:** `01-nlp-fundamentals.ipynb` introduces the concepts. This notebook uses `nlp_helpers.py` for preprocessing.


In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

from nlp_helpers import download_nltk_data, preprocess_text

download_nltk_data()
print('Setup complete.')


## 1. Labeled ticket sample

In production you would use thousands of historical tickets.


In [ ]:
# Simulated support tickets (realistic professional data)
tickets = pd.DataFrame({
    'text': [
        "I was charged twice for my order. Please refund the duplicate charge.",
        "My login password is not working. I cannot access my account.",
        "The package never arrived. Tracking says delivered but I didn't receive it.",
        "I need to update my payment method on file.",
        "The app crashes every time I open it on iOS.",
        "Wrong item was shipped. I ordered size M but received size L.",
        "My subscription renewed but I had cancelled it. Please reverse the charge.",
        "How do I reset my password? I forgot it.",
        "Delivery is 5 days late. When will I get my order?",
        "The website is very slow and times out frequently.",
        "I want to cancel my subscription and get a refund.",
        "Can't sync my data between devices. Getting error code 500.",
        "I was overcharged on my last invoice. Need a credit.",
        "Product is defective. I want a replacement or refund.",
        "Shipping address was wrong. Need to change it for next order.",
    ],
    'category': ['billing', 'technical', 'shipping', 'billing', 'technical', 'shipping',
                 'billing', 'technical', 'shipping', 'technical', 'billing', 'technical',
                 'billing', 'shipping', 'shipping']  # billing, technical, shipping
})

print("Support ticket categories:")
print(tickets['category'].value_counts())
print("\nSample tickets:")
print(tickets.head())

## 2. Train router

Same TF-IDF + logistic regression pattern as sentiment, but 3 classes.


In [ ]:
# Train a classifier for ticket routing
tickets['processed'] = tickets['text'].apply(preprocess_text)
vec = TfidfVectorizer(ngram_range=(1, 2))
X = vec.fit_transform(tickets['processed'])
y = tickets['category']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print("Ticket Routing Classification Report:")
print(classification_report(y_test, y_pred))

# Predict on new ticket (use same vectorizer!)
new_ticket = "My credit card was charged but I never received the product."
pred = clf.predict(vec.transform([preprocess_text(new_ticket)]))
print(f"\nNew ticket: '{new_ticket}'")
print(f"Predicted category: {pred[0]}")